In [128]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

In [129]:
df = pd.read_excel(r"C:\Users\LENOVO\OneDrive\Desktop\DATASETS\Telco_customer_churn.xlsx")

In [130]:
print(df.head())

   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   

                 Lat Long   Latitude   Longitude  Gender  ...        Contract  \
0  33.964131, -118.272783  33.964131 -118.272783    Male  ...  Month-to-month   
1   34.059281, -118.30742  34.059281 -118.307420  Female  ...  Month-to-month   
2  34.048013, -118.293953  34.048013 -118.293953  Female  ...  Month-to-month   
3  34.062125, -118.315709  34.062125 -118.315709  Female  ...  Month-to-month   
4  34.039224, -118.266293  34.039224 -118.266293    Male  ...  Month-to-month   

  Paperless Billing             Payment Method  Monthly Charges Tota

In [131]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [ ]:
df.shape

(7043, 33)

In [ ]:
# Convert target
# df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Ensure numeric columns
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Monthly Charges'] = pd.to_numeric(df['Monthly Charges'], errors='coerce')
df['Tenure Months'] = pd.to_numeric(df['Tenure Months'], errors='coerce')
df['CLTV'] = pd.to_numeric(df['CLTV'], errors='coerce')

In [ ]:
df.drop("Churn Reason", axis=1)

In [ ]:
df['Total Charges'].unique()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.columns

In [ ]:
drop_cols = ['CustomerID','Count','Country','State','City','Zip Code','Lat Long','Latitude','Longitude','Churn Label','Churn Score','Churn Reason']
df = df.drop(columns=drop_cols, errors='ignore')

In [ ]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [ ]:
df.columns

In [ ]:
X = df.drop('churn_value', axis=1)
y = df['churn_value']

In [ ]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
print("Numerical Columns:", list(num_cols))
print("Categorical Columns:", list(cat_cols))

In [ ]:
for col in cat_cols:
    X[col] = X[col].astype(str)


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, num_cols),
    ('cat', categorical_pipeline, cat_cols)
])


In [ ]:
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight='balanced',
        C=1.0
    ),
    "XGBClassifier": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=3
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}

# Threshold finder
def find_best_threshold(model_pipeline, X_test, y_test):
    y_prob = model_pipeline.predict_proba(X_test)[:, 1]
    
    best_f1 = 0
    best_threshold = 0.5
    best_metrics = {}

    for threshold in np.arange(0.1, 0.91, 0.05):
        y_pred = (y_prob >= threshold).astype(int)
        
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        acc = accuracy_score(y_test, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_metrics = {
                'Threshold': round(threshold, 2),
                'Accuracy': acc,
                'Precision': precision,
                'Recall': recall,
                'F1 Score': f1
            }

    return best_metrics

# Evaluate
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred_default = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    
    best_metrics = find_best_threshold(pipeline, X_test, y_test)

    results.append({
        'Model': name,
        'Default Accuracy': accuracy_score(y_test, y_pred_default),
        'Default F1': f1_score(y_test, y_pred_default),
        'Best Threshold': best_metrics['Threshold'],
        'Tuned Accuracy': best_metrics['Accuracy'],
        'Tuned Precision': best_metrics['Precision'],
        'Tuned Recall': best_metrics['Recall'],
        'Tuned F1 Score': best_metrics['F1 Score'],
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values(by='Tuned F1 Score', ascending=False))

In [ ]:
y_train

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Gradient Boosting")
plt.legend()
plt.show()


In [ ]:
# final_model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('model', GradientBoostingClassifier(
#         n_estimators=200,
#         learning_rate=0.05,
#         max_depth=3,
#         random_state=42
#     ))
# ])

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier())
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
joblib.dump(pipeline, "churn_pipeline.pkl")
print("Model saved successfully as churn_pipeline.pkl")